# Spike Classifier: Predicting Healthcare Sector Volatility from Public Health Signals

This notebook builds a binary Random Forest classifier to predict whether XLV (Health Care Select Sector ETF) weekly realized volatility will exceed a spike threshold defined as 1.5x the historical mean. Features are constructed from lagged public health signals (CDC influenza-like illness rates, Google Trends symptom queries) together with macro controls.

The analysis proceeds in four stages:

1. Constructing the binary spike target from realized volatility
2. Engineering lag features from health signals and macro variables
3. Training and evaluating an initial classifier on a temporal train/test split
4. Conducting a true out-of-sample test by training exclusively on pre-COVID data and evaluating on the COVID period (2020â€“2021)

The final stage produces an honest negative finding: a model with no knowledge of COVID-era dynamics cannot detect the crisis, motivating the regime detection approach developed in a separate notebook.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

master_df = pd.read_csv('../data/processed/master_weekly.csv', index_col=0, parse_dates=True)
print(master_df.shape)

(730, 19)


## Binary Target Construction

In [2]:
# Binary target: did XLV_vol exceed 1.5x its historical mean this week?
threshold = master_df['XLV_vol'].mean() * 1.5
master_df['spike'] = (master_df['XLV_vol'] > threshold).astype(int)

print(master_df['spike'].value_counts())
print(f"Spike rate: {master_df['spike'].mean()*100:.1f}%")

spike
0    621
1    109
Name: count, dtype: int64
Spike rate: 14.9%


The dataset contains 730 weekly observations. With the threshold set at 1.5x the historical mean of XLV realized volatility, 109 weeks (14.9%) are classified as spikes. The pronounced class imbalance â€” roughly 6 non-spike weeks for every spike â€” reflects the rarity of extreme market stress and motivates the use of `class_weight='balanced'` in the classifier.

## Feature Engineering (Lag Features)

In [3]:
features = pd.DataFrame(index=master_df.index)

health_signals = ['wili', 'trends_flu_symptoms', 'trends_fever', 'trends_shortness_of_breath']

# Generate 1- to 4-week lagged versions of each health signal
# Lagging ensures predictions use only information available before the target week
for signal in health_signals:
    for lag in range(1, 5):
        features[f'{signal}_lag{lag}'] = master_df[signal].shift(lag)

# Add contemporaneous macro controls; these are structural variables, not predictive health signals
features['cpi'] = master_df['cpi']
features['unemployment'] = master_df['unemployment']
features['interest_rate'] = master_df['interest_rate']
features['num_ili'] = master_df['num_ili']

features['spike'] = master_df['spike']
features = features.dropna()  # First 4 rows are NaN due to the maximum lag of 4

print(features.shape)
print(features['spike'].value_counts())

(726, 21)
spike
0    618
1    108
Name: count, dtype: int64


The feature matrix contains 726 rows (4 rows dropped because lag values are undefined for the first four weeks) and 21 columns. Sixteen columns are 1- to 4-week lagged versions of the four health signals, capturing the delayed relationship between illness trends and market volatility. The remaining four columns are contemporaneous macro controls. Using lagged signals enforces a proper causal structure: no future information is used to predict the current week's spike label.

## Initial Train/Test Split & Evaluation

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X = features.drop(columns=['spike'])
y = features['spike']

# shuffle=False preserves temporal order, preventing future data from leaking into training
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# class_weight='balanced' compensates for the 6:1 class imbalance by upweighting spike weeks
clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"Train spike rate: {y_train.mean()*100:.1f}%")
print(f"Test spike rate: {y_test.mean()*100:.1f}%")

Train size: 580, Test size: 146
Train spike rate: 15.7%
Test spike rate: 11.6%


In [5]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)[:, 1]  # Predicted probability for the spike class

roc_auc = roc_auc_score(y_test, y_pred_proba)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

ROC-AUC: 0.2207
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000

Confusion Matrix:
[[129   0]
 [ 17   0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.94       129
           1       0.00      0.00      0.00        17

    accuracy                           0.88       146
   macro avg       0.44      0.50      0.47       146
weighted avg       0.78      0.88      0.83       146



C:\Users\khavk\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\khavk\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\khavk\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\khavk\anaconda3\Lib\site-packages\sklea

Despite 88% overall accuracy, the model predicted zero spikes across the entire test set â€” it defaulted to the majority class throughout. A ROC-AUC of 0.22 (below the 0.5 random-chance baseline) indicates the model's probability scores are anti-correlated with true spike events, meaning the classifier is systematically more confident about non-spikes precisely when spikes are occurring. This reflects the difficulty of detecting rare events in a temporally ordered setting: patterns learned in the training period do not transfer cleanly to the tail end of the dataset, which includes the COVID period.

## Out-of-Sample COVID Test (Pre-COVID Train, COVID Test)

In [6]:
covid_mask = (features.index >= '2020-01-01') & (features.index <= '2021-01-01')
X_covid = X[covid_mask]
y_covid = y[covid_mask]

y_covid_pred = clf.predict(X_covid)
y_covid_proba = clf.predict_proba(X_covid)[:, 1]

roc_auc_covid = roc_auc_score(y_covid, y_covid_proba)
precision_covid = precision_score(y_covid, y_covid_pred, zero_division=0)
recall_covid = recall_score(y_covid, y_covid_pred, zero_division=0)
f1_covid = f1_score(y_covid, y_covid_pred, zero_division=0)

print(f"COVID Period ROC-AUC: {roc_auc_covid:.4f}")
print(f"COVID Period Precision: {precision_covid:.4f}")
print(f"COVID Period Recall: {recall_covid:.4f}")
print(f"COVID Period F1: {f1_covid:.4f}")
print(confusion_matrix(y_covid, y_covid_pred))

COVID Period ROC-AUC: 1.0000
COVID Period Precision: 1.0000
COVID Period Recall: 1.0000
COVID Period F1: 1.0000
[[35  0]
 [ 0 17]]


The perfect ROC-AUC of 1.0 reveals a data leakage problem. Because `clf` was trained on an 80/20 temporal split of the full dataset, the training set extends into the COVID period â€” the model has already seen COVID-era volatility patterns during training. Evaluating that same model on the COVID subset is not a true out-of-sample test. The following cell corrects this by training a new classifier (`clf2`) exclusively on pre-COVID data.

In [7]:
# New split: train on everything before COVID, test specifically on COVID
train_mask = features.index < '2020-01-01'
test_mask = (features.index >= '2020-01-01') & (features.index <= '2021-01-01')

X_train2 = X[train_mask]
y_train2 = y[train_mask]
X_test2 = X[test_mask]
y_test2 = y[test_mask]

# Retrain from scratch on pre-COVID data only â€” clf2 has never seen any COVID-era patterns
clf2 = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
clf2.fit(X_train2, y_train2)

y_pred2 = clf2.predict(X_test2)
y_proba2 = clf2.predict_proba(X_test2)[:, 1]

roc_auc2 = roc_auc_score(y_test2, y_proba2)
precision2 = precision_score(y_test2, y_pred2, zero_division=0)
recall2 = recall_score(y_test2, y_pred2, zero_division=0)
f1_2 = f1_score(y_test2, y_pred2, zero_division=0)

print(f"True Out-of-Sample COVID Test:")
print(f"ROC-AUC: {roc_auc2:.4f}")
print(f"Precision: {precision2:.4f}")
print(f"Recall: {recall2:.4f}")
print(f"F1: {f1_2:.4f}")
print(confusion_matrix(y_test2, y_pred2))

True Out-of-Sample COVID Test:
ROC-AUC: 0.4328
Precision: 0.0000
Recall: 0.0000
F1: 0.0000
[[35  0]
 [17  0]]


A model trained only on pre-COVID data achieved ROC-AUC of 0.43 on COVID-period data, meaning it failed to detect a genuinely novel crisis it had never seen before. This is an honest negative finding: historical-pattern-based classifiers are fundamentally limited at predicting unprecedented events. The confusion matrix shows all 17 spike weeks misclassified as non-spikes â€” the classifier predicted zero spikes across the entire COVID period. The scale and character of COVID-driven volatility had no precedent in the training data, so the learned decision boundaries provided no signal.

This motivates the regime detection approach explored separately, which aims to identify structural shifts in market behavior rather than predict individual spike events.

## Interpretation of Results

Across both the standard temporal split and the out-of-sample COVID test, the Random Forest spike classifier fails to achieve meaningful predictive performance:

- **Initial evaluation (80/20 temporal split):** ROC-AUC of 0.22 with no spikes predicted. Patterns learned from the training period do not generalize to the later test period.
- **COVID period test (leaky model):** Apparent ROC-AUC of 1.0, invalidated by COVID data appearing in the training set.
- **True out-of-sample COVID test:** ROC-AUC of 0.43 â€” below random chance â€” with the model predicting zero spikes during the most severe volatility period in the dataset.

These results establish a clear empirical bound: lag-feature-based classifiers trained on historical health signals cannot generalize to structurally novel crises. Volatility forecasting during unprecedented events requires a different approach â€” one that detects regime change rather than predicting individual outcomes. This motivates the unsupervised regime detection analysis in `regime_detection.ipynb`.